In [0]:
import org.apache.spark.sql.DataFrame
import java.time.LocalDate 
import java.time.format.DateTimeFormatter 
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

In [0]:

var sourcePath = dbutils.widgets.get("sourcePath")
var targetPath = dbutils.widgets.get("targetPath")
val header = dbutils.widgets.get("header")
val delimiter = dbutils.widgets.get("delimiter")
val filePrefix = dbutils.widgets.get("filePrefix")

//val sourcePath = "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/gestion_recursos/operacion_red/fatem/fallas_masivas_nuevas_manuales_fo/stage2/"
//val targetPath = "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/gestion_recursos/operacion_red/fatem/fallas_masivas_nuevas_manuales_fo/stage/"
//val header = "true"
//val delimiter = ";"
//val filePrefix = "fallas_masivas_nuevas_manuales_fo"

// Generate the file name with the current date (yyyyMMdd format)
val currentDate = LocalDate.now().format(DateTimeFormatter.ofPattern("yyyyMMdd"))
val finalFileName = s"${filePrefix}_$currentDate.csv"
val finalPath = s"$targetPath/$finalFileName"

// Function to delete the contents of the source folder
def deleteFolderContentsOnly(path: String): Unit = {
  try {
    val filesAndDirs = dbutils.fs.ls(path)
    if (filesAndDirs.nonEmpty) {
      filesAndDirs.foreach(file => dbutils.fs.rm(file.path, true))
      println(s"Se eliminó correctamente el contenido de la carpeta: $path")
    } else {
      println(s"La carpeta $path ya está vacía, no hay contenido que eliminar.")
    }
  } catch {
    case e: Exception => println(s"[ERROR] No se pudo eliminar el contenido de la carpeta $path: ${e.getMessage}")
  }
}

try {
  // Read all CSV files from the source folder
  val df: DataFrame = spark.read
    .option("header", header)
    .option("delimiter", delimiter)
    .option("inferSchema", "false")
    .csv(sourcePath)

  // Verify if the DataFrame has data
  if (df.count() > 0) {
    df.printSchema()
    println(s"Número total de registros: ${df.count()}")

    // Escribir en una carpeta temporal
    val tempPath = s"$targetPath/temp"
    df.repartition(1)
      .write
      .mode("overwrite")
      .option("delimiter", delimiter)
      .option("header", header)
      .csv(tempPath)

    // Get the name of the file generated by Spark
    val files = dbutils.fs.ls(tempPath).filter(_.name.startsWith("part-")).map(_.path)
    val generatedFile = files.head

    // Rename finalPath name
    dbutils.fs.mv(generatedFile, finalPath)

    // Delete the temporary folder
    dbutils.fs.rm(tempPath, true)

    println(s"Archivo combinado guardado en: $finalPath")
  } else {
    println("No se encontraron datos en los archivos CSV de la ruta de origen.")
  }

  // Delete the contents of the stage2 folder
  deleteFolderContentsOnly(sourcePath)

  // Notebook output
  dbutils.notebook.exit(finalPath)

} catch {
  case e: Exception =>
    println(s"Error al procesar los archivos: ${e.getMessage}")
    println("No se generó ningún archivo de salida debido a que no hay datos o ocurrió un problema.")
}